# Round 3 Comprehensive Analysis

This notebook performs:
1. **HYDROGEL_PACK** — mid-price analysis, MM vs EMA backtest
2. **VELVETFRUIT_EXTRACT** — level-based MR parameter sweep
3. **VEV_* Options** — vol-smile fitting, over/underpriced identification

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.stats import norm as _N

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

## Load Data

In [ ]:
df0 = pd.read_csv('data/prices_round_3_day_0.csv')
df1 = pd.read_csv('data/prices_round_3_day_1.csv')
df2 = pd.read_csv('data/prices_round_3_day_2.csv')

prices = pd.concat([df0, df1, df2], ignore_index=True)
prices['global_ts'] = prices['day'] * 1_000_000 + prices['timestamp']
print(f'Total rows: {len(prices):,}')
prices['product'].value_counts()

---
## 1. HYDROGEL_PACK Analysis

In [ ]:
hyd = prices[prices['product'] == 'HYDROGEL_PACK'].copy()
hyd['mid'] = hyd['mid_price']
print(f'Mean mid: {hyd["mid"].mean():.2f}')
print(f'Std mid:  {hyd["mid"].std():.2f}')
print(f'Min mid:  {hyd["mid"].min()}')
print(f'Max mid:  {hyd["mid"].max()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, d in enumerate([0, 1, 2]):
    sub = hyd[hyd['day'] == d]
    axes[i].plot(sub['timestamp'], sub['mid'], linewidth=0.5)
    axes[i].axhline(10000, color='r', linestyle='--', alpha=0.5)
    axes[i].set_title(f'HYDROGEL Day {d}')
plt.tight_layout()
plt.show()

### 1a. Backtest: EMA Cross vs Static MM

We compare a simple EMA crossover strategy vs the static MM approach.

In [ ]:
def backtest_ema_cross(df, fast=10, slow=30, pos_limit=200):
    """Simple EMA crossover: long when fast > slow, short when fast < slow."""
    mids = df['mid'].values
    n = len(mids)
    
    alpha_f = 2.0 / (fast + 1)
    alpha_s = 2.0 / (slow + 1)
    ema_f = mids[0]
    ema_s = mids[0]
    
    position = 0
    pnl = 0.0
    
    for i in range(1, n):
        ema_f = alpha_f * mids[i] + (1 - alpha_f) * ema_f
        ema_s = alpha_s * mids[i] + (1 - alpha_s) * ema_s
        
        target = pos_limit if ema_f > ema_s else -pos_limit
        delta = target - position
        
        if delta != 0:
            pnl -= delta * mids[i]
            position = target
    
    # close out
    pnl += position * mids[-1]
    return pnl

for d in [0, 1, 2]:
    sub = hyd[hyd['day'] == d].reset_index(drop=True)
    pnl = backtest_ema_cross(sub)
    print(f'Day {d} EMA Cross PnL: {pnl:,.0f}')

In [ ]:
def backtest_static_mm(df, pos_limit=200):
    """Simplified static MM: take when price deviates from wall_mid by >=1."""
    position = 0
    pnl = 0.0
    
    for _, row in df.iterrows():
        bid1, ask1 = row.get('bid_price_1'), row.get('ask_price_1')
        if pd.isna(bid1) or pd.isna(ask1):
            continue
        mid = (bid1 + ask1) / 2.0
        
        buy_room = pos_limit - position
        sell_room = pos_limit + position
        
        if ask1 <= mid - 1 and buy_room > 0:
            qty = min(abs(int(row.get('ask_volume_1', 0))), buy_room)
            if qty > 0:
                pnl -= qty * ask1
                position += qty
        
        if bid1 >= mid + 1 and sell_room > 0:
            qty = min(abs(int(row.get('bid_volume_1', 0))), sell_room)
            if qty > 0:
                pnl += qty * bid1
                position -= qty
    
    pnl += position * df['mid_price'].iloc[-1]
    return pnl

for d in [0, 1, 2]:
    sub = hyd[hyd['day'] == d].reset_index(drop=True)
    pnl = backtest_static_mm(sub)
    print(f'Day {d} Static MM PnL: {pnl:,.0f}')

---
## 2. VELVETFRUIT_EXTRACT Analysis

In [ ]:
vel = prices[prices['product'] == 'VELVETFRUIT_EXTRACT'].copy()
vel['mid'] = vel['mid_price']
print(f'Mean mid: {vel["mid"].mean():.2f}')
print(f'Std mid:  {vel["mid"].std():.2f}')
print(f'Min mid:  {vel["mid"].min()}')
print(f'Max mid:  {vel["mid"].max()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, d in enumerate([0, 1, 2]):
    sub = vel[vel['day'] == d]
    axes[i].plot(sub['timestamp'], sub['mid'], linewidth=0.5)
    axes[i].axhline(5240, color='g', linestyle='--', alpha=0.5, label='exit')
    axes[i].axhline(5230, color='b', linestyle='--', alpha=0.5, label='buy')
    axes[i].axhline(5265, color='r', linestyle='--', alpha=0.5, label='sell')
    axes[i].set_title(f'VELVET Day {d}')
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.show()

### 2a. Backtest: Level-Based Mean Reversion + Parameter Sweep

In [ ]:
def backtest_level_mr(df, buy_level, sell_level, exit_level, step_size, pos_limit=200):
    position = 0
    pnl = 0.0
    
    for _, row in df.iterrows():
        bid1, ask1 = row.get('bid_price_1'), row.get('ask_price_1')
        if pd.isna(bid1) or pd.isna(ask1):
            continue
        mid = (bid1 + ask1) / 2.0
        buy_room = pos_limit - position
        sell_room = pos_limit + position
        
        if mid <= buy_level:
            qty = min(step_size, buy_room)
            if qty > 0:
                pnl -= qty * ask1
                position += qty
        elif mid >= sell_level:
            qty = min(step_size, sell_room)
            if qty > 0:
                pnl += qty * bid1
                position -= qty
        elif position > 0 and mid >= exit_level:
            qty = min(position, step_size, sell_room)
            if qty > 0:
                pnl += qty * bid1
                position -= qty
        elif position < 0 and mid <= exit_level:
            qty = min(-position, step_size, buy_room)
            if qty > 0:
                pnl -= qty * ask1
                position += qty
    
    pnl += position * df['mid_price'].iloc[-1]
    return pnl

# Default params
total_pnl = 0
for d in [0, 1, 2]:
    sub = vel[vel['day'] == d].reset_index(drop=True)
    pnl = backtest_level_mr(sub, 5230, 5265, 5240, 25)
    total_pnl += pnl
    print(f'Day {d} Level MR PnL: {pnl:,.0f}')
print(f'Total: {total_pnl:,.0f}')

In [ ]:
# Parameter sweep
results = []
for buy_l in range(5225, 5236, 5):
    for sell_l in range(5260, 5271, 5):
        for exit_l in range(5235, 5246, 5):
            for step in [15, 20, 25, 30, 40, 50]:
                total = 0
                for d in [0, 1, 2]:
                    sub = vel[vel['day'] == d].reset_index(drop=True)
                    total += backtest_level_mr(sub, buy_l, sell_l, exit_l, step)
                results.append({
                    'buy': buy_l, 'sell': sell_l, 'exit': exit_l,
                    'step': step, 'pnl': total
                })

sweep_df = pd.DataFrame(results).sort_values('pnl', ascending=False)
print('Top 10 parameter combinations:')
sweep_df.head(10)

---
## 3. VEV_* Options Analysis

In [ ]:
OPTION_SYMBOLS = [
    'VEV_4000', 'VEV_4500', 'VEV_5000', 'VEV_5100',
    'VEV_5200', 'VEV_5300', 'VEV_5400', 'VEV_5500',
    'VEV_6000', 'VEV_6500'
]

DAYS_PER_YEAR = 365

def bs_call(S, K, TTE, sigma):
    if TTE <= 0 or sigma <= 0:
        return max(S - K, 0), 1.0 if S > K else 0.0
    d1 = (np.log(S/K) + 0.5 * sigma**2 * TTE) / (sigma * TTE**0.5)
    d2 = d1 - sigma * TTE**0.5
    return S * _N.cdf(d1) - K * _N.cdf(d2), _N.cdf(d1)

def implied_vol_newton(S, K, TTE, market_price, tol=1e-6, max_iter=100):
    """Newton's method IV solver."""
    sigma = 0.2
    for _ in range(max_iter):
        price, _ = bs_call(S, K, TTE, sigma)
        d1 = (np.log(S/K) + 0.5 * sigma**2 * TTE) / (sigma * TTE**0.5)
        vega = S * _N.pdf(d1) * TTE**0.5
        if abs(vega) < 1e-12:
            break
        sigma -= (price - market_price) / vega
        if sigma <= 0:
            sigma = 0.001
        if abs(price - market_price) < tol:
            break
    return sigma

In [ ]:
# Compute implied vol and moneyness for each option at each timestamp
vel_ts = prices[prices['product'] == 'VELVETFRUIT_EXTRACT'][['day', 'timestamp', 'mid_price']].copy()
vel_ts = vel_ts.rename(columns={'mid_price': 'S'})

# TTE map: day 0 → TTE = 8/365, day 1 → 7/365, day 2 → 6/365
tte_map = {0: 8.0/365, 1: 7.0/365, 2: 6.0/365}

all_iv_data = []
for sym in OPTION_SYMBOLS:
    K = int(sym.split('_')[-1])
    opt_df = prices[prices['product'] == sym][['day', 'timestamp', 'mid_price']].copy()
    opt_df = opt_df.rename(columns={'mid_price': 'V'})
    
    merged = opt_df.merge(vel_ts, on=['day', 'timestamp'], how='inner')
    
    for _, row in merged.iterrows():
        S = row['S']
        V = row['V']
        day = row['day']
        ts = row['timestamp']
        
        base_tte = tte_map.get(day, 7.0/365)
        tte = base_tte - (ts / 100.0 / 10_000.0) / 365
        if tte <= 0:
            continue
        
        intrinsic = max(S - K, 0)
        if V <= intrinsic:
            continue
        
        try:
            iv = implied_vol_newton(S, K, tte, V)
            if iv <= 0 or iv > 3:
                continue
            m = np.log(K / S) / np.sqrt(tte)
            all_iv_data.append({
                'day': day, 'timestamp': ts, 'strike': K,
                'S': S, 'V': V, 'tte': tte, 'm': m, 'iv': iv
            })
        except:
            pass

iv_df = pd.DataFrame(all_iv_data)
print(f'IV data points: {len(iv_df):,}')
iv_df.head()

In [ ]:
# Fit vol smile: v(m) = a*m^2 + b*m + c
if len(iv_df) > 10:
    coeffs = np.polyfit(iv_df['m'], iv_df['iv'], 2)
    print(f'Vol smile coefficients: {coeffs}')
    
    m_range = np.linspace(iv_df['m'].min(), iv_df['m'].max(), 200)
    fitted_iv = np.polyval(coeffs, m_range)
    
    plt.figure(figsize=(10, 5))
    plt.scatter(iv_df['m'], iv_df['iv'], s=1, alpha=0.3, label='Observed IV')
    plt.plot(m_range, fitted_iv, 'r-', linewidth=2, label='Fitted parabola')
    plt.xlabel('Moneyness m = log(K/S) / sqrt(TTE)')
    plt.ylabel('Implied Volatility')
    plt.title('Volatility Smile')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('Not enough data for vol smile fitting')
    coeffs = [0.27362531, 0.01007566, 0.14876677]

In [ ]:
# Compute theo price from fitted IV and compare to market
if len(iv_df) > 0:
    iv_df['fitted_iv'] = np.polyval(coeffs, iv_df['m'])
    
    def theo_price(row):
        p, _ = bs_call(row['S'], row['strike'], row['tte'], row['fitted_iv'])
        return p
    
    iv_df['theo'] = iv_df.apply(theo_price, axis=1)
    iv_df['mispricing'] = iv_df['V'] - iv_df['theo']
    
    print('Average mispricing by strike:')
    print(iv_df.groupby('strike')['mispricing'].agg(['mean', 'std', 'count']).round(3))

In [ ]:
# Base IV over time (IV at m=0)
if len(iv_df) > 0:
    base_iv_series = []
    for (day, ts), grp in iv_df.groupby(['day', 'timestamp']):
        if len(grp) >= 3:
            c = np.polyfit(grp['m'], grp['iv'], 2)
            base_iv_series.append({'day': day, 'timestamp': ts, 'base_iv': c[2]})
    
    base_iv_df = pd.DataFrame(base_iv_series)
    if len(base_iv_df) > 0:
        base_iv_df['global_ts'] = base_iv_df['day'] * 1_000_000 + base_iv_df['timestamp']
        plt.figure(figsize=(14, 4))
        plt.plot(base_iv_df['global_ts'], base_iv_df['base_iv'], linewidth=0.5)
        plt.title('Base IV (at m=0) Over Time')
        plt.xlabel('Global Timestamp')
        plt.ylabel('Base IV')
        plt.grid(True, alpha=0.3)
        plt.show()
        print(f'Base IV mean: {base_iv_df["base_iv"].mean():.4f}')
        print(f'Base IV std:  {base_iv_df["base_iv"].std():.4f}')

---
## Summary

### Key Findings

1. **HYDROGEL_PACK** oscillates tightly around 10,000. Static MM captures spread effectively; EMA crossover tends to lose due to mean-reverting nature.

2. **VELVETFRUIT_EXTRACT** is range-bound (~5200–5300). Level-based MR with buy@5230, sell@5265, exit@5240 produces strong positive PnL. SMA crossover strategies consistently lose.

3. **VEV_* Options** show a well-defined volatility smile. Fitting a parabola to IV vs moneyness allows identification of over/underpriced options relative to the fitted curve. IV scalping on ATM strikes and MR on deep ITM strikes are viable strategies.